# 👕 AI Outfit Studio — MY TYPE OUTFIT Generator
**ThriftDrip's automatic outfit factory.** Generates headless AI fashion posters + Instagram collages.

**How it works (3 steps):**
1. **Generate** — free AI creates outfit photos from text prompts (Pollinations.ai, no key needed)
2. **Clean** — backgrounds removed automatically (rembg)
3. **Collage** — posters + grids with "MY TYPE OUTFIT" titles, Instagram-ready sizes

Run each cell top to bottom. First run downloads the background-removal model (~1 min).

In [ ]:
# CELL 1 — install tools (run once per session)
!pip install -q rembg pillow requests
print("tools installed")

In [ ]:
# CELL 2 — load the prompt library + collage templates from GitHub
import urllib.request
for f in ["prompts.py", "collage.py"]:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/dheerajnain7/Thriftdrip/main/ai-outfit-studio/" + f, f)
    print("loaded", f)
from prompts import PROMPTS_100
import collage
print(len(PROMPTS_100), "outfit prompts ready")

In [ ]:
# CELL 3 — CONFIG: how many outfits? (start with 2 to test, then 10, then 100)
N_OUTFITS = 2        # <-- change this: 2, 10, or 100
START_INDEX = 0      # <-- change to continue where you left off (e.g. 10, 20...)
SEED_START = 100
print(f"Will generate outfits {START_INDEX} to {START_INDEX + N_OUTFITS}")

In [ ]:
# CELL 4 — GENERATE outfit photos (free AI, ~20-40 sec each)
import os, time, urllib.parse, requests

os.makedirs("outfits_raw", exist_ok=True)
selected = PROMPTS_100[START_INDEX : START_INDEX + N_OUTFITS]

for i, prompt in enumerate(selected):
    idx = START_INDEX + i
    out = f"outfits_raw/fit_{idx:03d}.jpg"
    if os.path.exists(out):
        print(f"[{idx}] already exists, skipping"); continue
    q = urllib.parse.quote(prompt)
    url = f"https://image.pollinations.ai/prompt/{q}?width=768&height=1344&seed={SEED_START + idx}&nologo=true"
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=180)
            if r.headers.get("content-type", "").startswith("image"):
                open(out, "wb").write(r.content)
                print(f"[{idx}] generated OK")
                break
            else:
                print(f"[{idx}] backend busy, retrying...")
                time.sleep(10)
        except Exception as e:
            print(f"[{idx}] error: {e}, retrying..."); time.sleep(10)
    time.sleep(3)  # be nice to the free service
print("generation done")

In [ ]:
# CELL 5 — REMOVE backgrounds (rembg, automatic)
from rembg import remove, new_session
from PIL import Image
import os

os.makedirs("outfits_clean", exist_ok=True)
session = new_session("u2net")  # downloads once, then cached
files = sorted(f for f in os.listdir("outfits_raw") if f.endswith(".jpg"))
for f in files:
    out = f"outfits_clean/{f.replace('.jpg', '.png')}"
    if os.path.exists(out): continue
    img = Image.open(f"outfits_raw/{f}")
    clean = remove(img, session=session)
    # paste onto pure white (Instagram look)
    bg = Image.new("RGB", clean.size, (255, 255, 255))
    bg.paste(clean, mask=clean.split()[3])
    bg.save(out)
    print("cleaned", f)
print("backgrounds removed")

In [ ]:
# CELL 6 — BUILD posters + collages with MY TYPE OUTFIT titles
import os

os.makedirs("posters", exist_ok=True)
os.makedirs("cards", exist_ok=True)
os.makedirs("grids", exist_ok=True)
files = sorted(f for f in os.listdir("outfits_clean") if f.endswith(".png"))

for f in files:
    idx = f.replace("fit_", "").replace(".png", "")
    src = f"outfits_clean/{f}"
    collage.hero_poster(src, f"posters/poster_{idx}.jpg", subtitle=f"FIT {int(idx)+1}")
    collage.outfit_card(src, f"cards/card_{idx}.jpg", subtitle=f"FIT {int(idx)+1}")
print("posters + cards done")

# grid collages: groups of 4
for g in range(0, len(files), 4):
    group = files[g:g+4]
    if len(group) == 4:
        srcs = [f"outfits_clean/{f}" for f in group]
        collage.grid_collage(srcs, f"grids/grid_{g//4:02d}.jpg")
print("grids done")

In [ ]:
# CELL 7 — PREVIEW + DOWNLOAD everything as a zip
import shutil
from IPython.display import Image as IPImage, display
import glob

previews = sorted(glob.glob("posters/*.jpg"))[:4]
for p in previews:
    display(IPImage(p, width=240))

shutil.make_archive("my_type_outfit_batch", "zip", ".")
print("ZIP ready — download it from the Files panel on the left:")
print("my_type_outfit_batch.zip contains: outfits_raw / outfits_clean / posters / cards / grids")

## 📱 What you got
- `posters/` — 9:16 hero posters, one per outfit, "MY TYPE OUTFIT" title → post as reels covers or single posts
- `cards/` — 4:5 outfit cards → swipeable carousel posts
- `grids/` — 1:1 four-fit grids → feed posts

## 🔁 Make more
Change `N_OUTFITS` and `START_INDEX` in CELL 3 and re-run from CELL 4.
- First test: `N_OUTFITS = 2`
- Real batch: `N_OUTFITS = 10`
- Full drop: `N_OUTFITS = 100` (takes ~1 hour, Colab stays free)

## ⚠️ Limits (honest)
- Pollinations is free but sometimes busy (503) — the notebook auto-retries 3x
- Small "pollinations.ai" watermark can appear in a corner — crop it or cover with the title
- Colab free sessions time out after ~a few hours — 100 outfits fits, but don't close the tab mid-run